# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [52]:
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [53]:
 !pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [54]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import ChatOpenAI

In [55]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [67]:
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    api_key=OPENAI_API_KEY
)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [57]:
doc_path =  "./datasets/state_of_the_union.txt"

In [58]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [68]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [60]:
from langchain_community.document_loaders import WebBaseLoader

In [61]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [69]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [63]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [64]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [70]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [71]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: Biden said he had nominated Judge Ketanji Brown Jackson to the Supreme Court, calling her “one of our nation’s top legal minds” and saying she would continue Justice Stephen Breyer’s “legacy of excellence.”
Thought:I now know the final answer.  
Final Answer: Biden said he had nominated Judge Ketanji Brown Jackson to the Supreme Court, calling her “one of our nation’s top legal minds” and stating that she would continue Justice Stephen Breyer’s “legacy of excellence.”

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': 'Biden said he had nominated Judge Ketanji Brown Jackson to the Supreme Court, calling her “one of our nation’s top legal minds” and stating that she would continue Justice Stephen Breyer’s “legacy of excellence.”'}

In [72]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and advantages of Ruff and Flake8 to provide a comprehensive answer about why one might choose Ruff over Flake8. 
Action: Ruff QA System
Action Input: "Why should someone use Ruff over Flake8 for Python linting?"
Observation: Ruff is appealing when you want a simpler, more integrated linting setup:

- **It bundles many popular Flake8-plugin rules** as built-in rules, so you can avoid managing separate plugins.
- **Its rule groups are easy to select** and use distinct prefixes to reduce code conflicts—for example, Ruff uses `TID252` for a tidy-imports rule rather than reusing a potentially conflicting prefix.
- **It can automatically fix some violations**, which can make cleanup quicker.

There are trade-offs: Ruff doesn’t support custom or third-party lint rules, and it doesn’t include every opinionated `flake8-bugbear` rule. If your setup relies on those, check Ruff’s rule coverage before switching.
Thought:I now know the final answer.  
Final Answer: Ru

{'input': 'Why use ruff over flake8?',
 'output': "Ruff is preferred over Flake8 for Python linting because it bundles many popular Flake8-plugin rules as built-in rules, simplifying setup by avoiding the need to manage separate plugins. It also offers easy selection of rule groups with distinct prefixes to minimize code conflicts and can automatically fix some violations, speeding up the cleanup process. However, it does not support custom or third-party lint rules and may lack some specific rules from Flake8 plugins, so users should verify Ruff's rule coverage before switching."}

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [73]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [74]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [75]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence."}

In [76]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and benefits of Ruff and Flake8 to provide a comprehensive answer about why one might choose Ruff over Flake8. 
Action: Ruff QA System
Action Input: "Why should someone use Ruff over Flake8 for Python linting?"
Observation: Someone might choose to use Ruff over Flake8 for Python linting for several reasons:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting coverage.

3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during the development process.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.

5. **Simplified Configuration**: R

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 for Python linting for several reasons:\n\n1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting coverage.\n\n3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during the development process.\n\n4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.\n\n5. **Simplified Configuration**: Ruff uses different rule codes and prefixes to minimize conflicts across plugins, allowing users to toggle rules on or off more easily.\n\n6. **Complementary Use with Type Checkers**: Ruff is des

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [77]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [78]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [79]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what tool Ruff uses to run over Jupyter Notebooks and whether the president mentioned that tool in the State of the Union address. I will first check the Ruff QA System for information about the tool used by Ruff.

Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought:Now that I know Ruff uses nbQA to run over Jupyter Notebooks, I need to check if the president mentioned nbQA in the State of the Union address.

Action: State of Union QA System  
Action Input: Did the president mention nbQA in the State of the Union address?  
Observation: I don't know.
Thought:I need to find out if the president mentioned nbQA in the State of the Union address. Since I don't have the information yet, I will try again to get the relevant details from the State of Union QA System.

Action: State of Union QA System  
Action Inpu

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks, but it is unclear if the president mentioned nbQA or any related tools in the State of the Union address.'}

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.



In [80]:
hockey_text = '''
ice hockey is a team sport played on ice.
each team has six players on the ice, including the goaltender
a standard game consists of three periods of 20 minuted
the nhl is the national hockey league in north america
the stanley cup is awarded to the nhl playoff champion
alex ovechkin is known as the great one.
'''

with open('hockey.txt', 'w') as f:
    f.write(hockey_text)

In [81]:
hockey_loader = TextLoader('hockey.txt')
hockey_docs = hockey_loader.load()

hockey_texts = text_splitter.split_documents(hockey_docs)

hockey_db = Chroma.from_documents(
    hockey_texts,
    embeddings,
    collection_name='hockey_knowledge'
)

hockey_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=hockey_db.as_retriever()
)

In [82]:
tools = [
    Tool(
        name='hockey qa system',
        func=hockey_qa.run,
        description='use this tool for questions about ice hockey, nhl rules, players and the stanley cup'
    ),
    Tool(
        name='ruff qa system',
        func=ruff.run,
        description='use this tool for questions about the ruff python linter'
    )
]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

In [83]:
agent.invoke('how many periods are in an ice hockey game?')

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


An ice hockey game consists of multiple periods, and I need to confirm the exact number. 
Action: hockey qa system
Action Input: "how many periods are in an ice hockey game?"
Observation: A standard ice hockey game consists of three periods.
Thought:I now know the final answer.  
Final Answer: A standard ice hockey game consists of three periods.

> Finished chain.


{'input': 'how many periods are in an ice hockey game?',
 'output': 'A standard ice hockey game consists of three periods.'}

In [84]:
agent.invoke('what is ruff used for in python?')

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Ruff is a linter for Python that helps developers identify and fix issues in their code. It can enforce coding standards, detect potential bugs, and improve code quality. To provide a more detailed answer, I will use the ruff qa system to gather specific information about its features and functionalities.  
Action: ruff qa system  
Action Input: "What is Ruff used for in Python?"  
Observation: Ruff is used as a linter and formatter for Python code. It can replace tools like Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade. Ruff helps in linting code for any Python version from 3.7 onwards, ensuring code quality and style consistency.
Thought:I now know the final answer.  
Final Answer: Ruff is used as a linter and formatter for Python code. It can replace tools like Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade. Ruff helps in linting code for any Python version from 3.7 onwards, ensuring code quality and style consistency.

{'input': 'what is ruff used for in python?',
 'output': 'Ruff is used as a linter and formatter for Python code. It can replace tools like Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade. Ruff helps in linting code for any Python version from 3.7 onwards, ensuring code quality and style consistency.'}

In [85]:
agent.invoke('how many periods are in ice hockey, and what is ruff used for in python?')

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to answer two parts of the question: the number of periods in ice hockey and the purpose of the ruff linter in Python. I'll first address the ice hockey part using the hockey QA system, and then I'll use the ruff QA system for the Python linter part.

Action: hockey qa system  
Action Input: "how many periods are in ice hockey?"  
Observation: A standard game of ice hockey consists of three periods.
Thought:I have the answer for the ice hockey part. Now, I need to find out what ruff is used for in Python.

Action: ruff qa system  
Action Input: "What is ruff used for in Python?"  
Observation: Ruff is used as a linter and formatter for Python code. It can replace tools like Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade. Ruff helps in identifying and fixing code issues, ensuring code quality, and formatting code according to specified styles.
Thought:I now know the final answer, which includes both parts of the question regarding ice hockey and th

{'input': 'how many periods are in ice hockey, and what is ruff used for in python?',
 'output': 'A standard game of ice hockey consists of three periods. Ruff is used as a linter and formatter for Python code, helping to identify and fix code issues and ensuring code quality.'}

In [86]:
tools[0].return_direct = True

agent_direct = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent_direct.invoke('how many periods are in an ice hockey game?')

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


An ice hockey game consists of multiple periods, and I need to confirm the exact number. 
Action: hockey qa system
Action Input: "how many periods are in an ice hockey game?"
Observation: A standard ice hockey game consists of three periods.


> Finished chain.


{'input': 'how many periods are in an ice hockey game?',
 'output': 'A standard ice hockey game consists of three periods.'}